In [15]:
# imports
import pandas as pd
import torch
from torch import nn
import numpy as np

%load_ext autoreload
%autoreload 2
import models as m
import evaluation as ev

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# set the device
if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [3]:
# fixed values, as was set in 06_models
dropout = 0.2
n_classes = 3
batch_size = 64
learning_rate = 1e-3
seed = 7

# refit settings for the six conditions
refit_settings = {
    "text": {
        "hidden_width": 512,
        "epochs": 8,
    },
    "image": {
        "hidden_width": 256,
        "epochs": 42,
    },
    "early": {
        "hidden_width": 128,
        "epochs": 31,
    },
    "intermediate": {
        "hidden_width": 512,
        "epochs": 49,
    },
    "late": {
        "hidden_width": 128,
        "epochs": 30,
    },
    "learned": {
        "hidden_width": 256,
        "epochs": 17,
    },
}

In [7]:
# load the training data with labels
train_with_target_full = pd.read_csv('data/train_with_target.csv', usecols=['external_code', 'label_train'])
has_training_label = train_with_target_full['label_train'].notna()
training_labels = train_with_target_full[has_training_label].astype({'label_train': 'int64'})

# should get the 5080 prods
print(train_with_target_full.shape)
# all products not in the buffer zone, 3817
print(training_labels.shape)
# roughly equal value counts for each class
print(training_labels['label_train'].value_counts())

(5080, 2)
(3817, 2)
label_train
0    1278
2    1270
1    1269
Name: count, dtype: int64


In [ ]:
# load product IDs
training_codes = training_labels['external_code'].tolist()

# load parquets
txt_parquet_all  = pd.read_parquet('data/clip_text.parquet').set_index('external_code')
img_parquet_all  = pd.read_parquet('data/clip_image.parquet').set_index('external_code')

training_txt = txt_parquet_all.loc[training_codes]
training_img = img_parquet_all.loc[training_codes]

print("Text: ", training_txt.shape, training_txt.dtypes.unique())
print("Image: ", training_img.shape, training_img.dtypes.unique())
print(training_labels['label_train'].value_counts().sort_index())
print(training_labels['label_train'].dtype)

Text:  (3817, 768) [dtype('float32')]
Image:  (3817, 768) [dtype('float32')]
label_train
0    1278
1    1269
2    1270
Name: count, dtype: int64
int64


In [11]:
# load test data
test_labels = pd.read_csv('data/test_with_target.csv', usecols=['external_code', 'label_eval', 'in_buffer'])

test_codes = test_labels['external_code'].tolist()

test_txt = txt_parquet_all.loc[test_codes]
test_img = img_parquet_all.loc[test_codes]

print("Test shape: ", test_labels.shape)
print("Missing labels: ", test_labels['label_eval'].isna().sum())
print(test_labels['label_eval'].value_counts().sort_index())

Test shape:  (497, 3)
Missing labels:  0
label_eval
0     83
1    286
2    128
Name: count, dtype: int64


In [12]:
# convert to tensors
training_txt_tensor = torch.as_tensor(training_txt.to_numpy()).to(device)
training_img_tensor = torch.as_tensor(training_img.to_numpy()).to(device)

training_label_tensor = torch.as_tensor(
    training_labels['label_train'].to_numpy(), dtype=torch.int64
).to(device)

test_txt_tensor = torch.as_tensor(test_txt.to_numpy()).to(device)
test_img_tensor = torch.as_tensor(test_img.to_numpy()).to(device)
test_label_tensor = torch.as_tensor(
    test_labels['label_eval'].to_numpy(), dtype=torch.int64
).to(device)

# check everything transformed correctly
for tensor_name, tensor in [
    ("training text: ", training_txt_tensor),
    ("training image: ", training_img_tensor),
    ("training labels: ", training_label_tensor),
    ("test_txt: ", test_txt_tensor),
    ("test_img: ", test_img_tensor),
    ("test_labels: ", test_label_tensor),
]:
    print(f'{tensor_name} : {tuple(tensor.shape)} {tensor.dtype} {tensor.device} ')

/var/folders/k7/bvctb7lj5wv34r9yjkch19340000gn/T/ipykernel_73586/4054617102.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  training_txt_tensor = torch.as_tensor(training_txt.to_numpy()).to(device)


training text:  : (3817, 768) torch.float32 mps:0 
training image:  : (3817, 768) torch.float32 mps:0 
training labels:  : (3817,) torch.int64 mps:0 
test_txt:  : (497, 768) torch.float32 mps:0 
test_img:  : (497, 768) torch.float32 mps:0 
test_labels:  : (497,) torch.int64 mps:0 


In [13]:
def refit_one_condition (condition, hidden_width, epochs, train_data):
    torch.manual_seed(seed)

    epoch_losses = []
    input_dim = train_data['text'].shape[1]

    config = {
        'condition': condition,
        'input_dim': input_dim,
        'hidden_width': hidden_width,
        'dropout': dropout,
        'n_classes': n_classes,
    }

    model = m.ConditionModel(config).to(train_data['text'].device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0,
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    loss_function = nn.CrossEntropyLoss()
    n_train = train_data['text'].shape[0]

    for epoch in range(1, epochs +1):
        model.train()
        running_loss = 0.0

        shuffled = torch.randperm(n_train, device=train_data['text'].device)

        for start in range(0, n_train, batch_size):
            batch_rows = shuffled[start:start + batch_size]

            batch_text = train_data['text'][batch_rows]
            batch_image = train_data['image'][batch_rows]
            batch_labels = train_data['labels'][batch_rows]

            optimizer.zero_grad()
            logits = model(batch_text, batch_image)
            loss = loss_function(logits, batch_labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(batch_rows)

        train_loss = running_loss / n_train
        epoch_losses.append(train_loss)

    return model, epoch_losses

training_data = {
    'text': training_txt_tensor,
    'image': training_img_tensor,
    'labels': training_label_tensor
}

In [14]:
# two empty dictionaries to hold the results
refit_models = {}
refit_losses = {}

for condition, settings in refit_settings.items():
    model, epoch_losses = refit_one_condition(condition, settings['hidden_width'], settings['epochs'], training_data)

    refit_models[condition] = model
    refit_losses[condition] = epoch_losses

    print(f'{condition:<15} first loss: {epoch_losses[0]:.4f}  last loss: {epoch_losses[-1]:.4f}')

text            first loss: 1.0741  last loss: 0.9514
image           first loss: 1.0628  last loss: 0.2657
early           first loss: 1.0583  last loss: 0.3866
intermediate    first loss: 1.0528  last loss: 0.1164
late            first loss: 1.0587  last loss: 0.3707
learned         first loss: 1.0587  last loss: 0.5502


In [16]:
# two empty dictionaries to hold the results
# from the test data
test_predictions = {}
test_gates = {}

for condition, model in refit_models.items():
    model.eval()

    with torch.no_grad():
        logits, gate_values = model(test_txt_tensor, test_img_tensor, return_gate=True)

    predictions = logits.argmax(dim=1).cpu().numpy()
    test_predictions[condition] = predictions

    # gate only for learned
    if condition == 'learned':
        test_gates[condition] = gate_values.squeeze(1).cpu().numpy()

    # order 0 flat, 1 slow, 2 fast
    class_counts = np.bincount(predictions, minlength=3)
    print(f'{condition:<15} n={len(predictions)} predicted flat/slow/fast: {class_counts}')

learned_gates = test_gates['learned']
print(
    f'gates: n={len(learned_gates)} min {learned_gates.min():.4f}'
    f'max {learned_gates.max():.4f} mean {learned_gates.mean():.4f}'
      )

text            n=497 predicted flat/slow/fast: [162 180 155]
image           n=497 predicted flat/slow/fast: [167 169 161]
early           n=497 predicted flat/slow/fast: [135 139 223]
intermediate    n=497 predicted flat/slow/fast: [154 187 156]
late            n=497 predicted flat/slow/fast: [153 200 144]
learned         n=497 predicted flat/slow/fast: [127 153 217]
gates: n=497 min 0.0238max 0.8364 mean 0.3563
